# Data profiling: amazon-purchases.csv

**Source:** Berke et al. (2023), *Open e-commerce 1.0*, Harvard Dataverse,
https://doi.org/10.7910/DVN/YGLYDY — license CC0 1.0. Only the purchases file is used.

This notebook follows a fixed **profiling checklist** of 8 questions.
Run it top to bottom with **Run All**. Findings and decisions are recorded in
`docs/profiling.md` and `docs/decisions.md`.

| # | Question | Main function |
|---|---|---|
| 1 | Is every row unique? | `duplicated()` |
| 2 | How many different values per column? | `nunique()` |
| 3 | Which values are most common? | `value_counts()` |
| 4 | What do the numbers look like? | `describe()`, `sort_values()` |
| 5 | Are missing values connected? | `isna()`, `crosstab()` |
| 6 | Are there gaps over time? | `resample()`, `plot()` |
| 7 | How do numbers relate to each other? | `corr()` |
| 8 | Is the text clean? | `str.strip()`, `str.upper()` |

## Setup
Load the file (downloaded and MD5-verified by `scripts/download_data.py`) and
convert the dates. This cell makes the notebook independent of run order.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/amazon-purchases.csv")
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["spend"] = df["Purchase Price Per Unit"] * df["Quantity"]

df.shape

In [ ]:
df.head(10)

## Missing values per column
`isna().sum()` counts empty values across **all** rows (not only the first 10).

**Finding:** ~4.8 % of rows have no title / category, ~4.7 % no state.
**Decision 002:** incomplete but real rows are kept and labelled "Unknown".

In [ ]:
df.isna().sum()

## Check 1 — Is every row unique?
`duplicated()` marks rows that are exact copies of an earlier row.
`keep=False` marks **all** copies, so each pair can be seen together.

**Finding:** 11,624 identical rows (0.6 %). There is no order number, so they cannot
be told apart from real repeat purchases (two gift cards, two teas, two frames).
**Decision 003:** kept and documented.

In [ ]:
df.duplicated().sum()

In [ ]:
dups = df[df.duplicated(keep=False)]
dups.sort_values(["Survey ResponseID", "Order Date", "Title"]).head(10)

### Gift cards: a possible double count
Amazon's export lists **what** was bought, not **how** it was paid. A $200 balance
reload and the $200 of products later paid with it both appear as purchases.
The source has no payment method, so this cannot be confirmed.

**Decision 004:** gift cards are reported separately and excluded from "spending on products".

In [ ]:
gift = df["Category"].isin(["GIFT_CARD", "Ecard Gift Certificate"])
gift_spend, total_spend = df.loc[gift, "spend"].sum(), df["spend"].sum()
gift_spend, total_spend, gift_spend / total_spend

## Check 2 — How many different values per column?
Few values → a category (good for filters). Almost as many as rows → an identifier
or free text. Expected numbers are a quality signal.

**Findings:** exactly 5,027 customers (matches the paper); 52 states = 50 + DC +
Puerto Rico; 1,871 categories — far too many for a chart.

In [ ]:
df.nunique()

In [ ]:
# The rarest values are where odd ones hide
df["Shipping Address State"].value_counts().tail(5)

### How concentrated is spending across categories? (Pareto / 80-20)
**Finding:** 397 categories make 80 % of spending, 661 make 90 %: spending is spread out,
so categories are grouped with **keyword rules** (in dbt), not mapped by hand.

In [ ]:
by_cat = df.groupby("Category")["spend"].sum().sort_values(ascending=False)
share = by_cat.cumsum() / by_cat.sum()
(share <= 0.8).sum(), (share <= 0.9).sum()

In [ ]:
by_cat.head(40)

## Check 3 — Which values are most common?
`value_counts()` counts how often each value appears.

**Findings:** 94 % of purchases are one piece. Purchases per customer: median 232,
average 368, maximum 5,415 — a few heavy buyers pull the average up.

In [ ]:
df["Quantity"].value_counts().head(5)

In [ ]:
per_customer = df["Survey ResponseID"].value_counts()
per_customer.describe()

### Reading `describe()`: the line of people
Sort all customers from fewest to most purchases and stand them in a line:
**min** = first, **25 %** = a quarter of the way, **50 %** = the middle (median),
**75 %** = three quarters, **max** = last. **mean** = the average, **std** = how spread out.

**Skewness** puts the average-vs-median gap into one number:
about 0 = balanced, above 1 = strongly skewed (a long tail of big values).
**Finding:** ≈ 3 → report the **median** customer, not the average.

In [ ]:
per_customer.skew()

## Check 4 — What do the numbers look like?
**Findings:** prices from $0.01 to $6,398.95, median $13.99, average $22.66.
Every outlier was checked in context and is real: 339 photo prints, $0.01 samples /
SIM cards / used books, a 96 TB storage system. An outlier is not automatically an error.

In [ ]:
df[["Purchase Price Per Unit", "Quantity"]].describe()

In [ ]:
df.sort_values("Quantity", ascending=False).head(5)

In [ ]:
cheap = df[df["Purchase Price Per Unit"] <= 0.01]
len(cheap), cheap["Title"].head(10)

In [ ]:
cols = ["Order Date", "Purchase Price Per Unit", "Title", "Category"]
df.sort_values("Purchase Price Per Unit", ascending=False).head(5)[cols]

### Dates
Data was collected Nov 2022 – Mar 2023, so nothing can be later than March 2023.

**Findings:** 1 row dated 2024-08-15 (a protection plan) → invalid, rejected with a reason.
2023 is only partly covered → **decision 005:** yearly trends use 2018–2022.
Purchases +54 % from 2019 to 2020 (COVID) — a real pattern.

In [ ]:
df["Order Date"].min(), df["Order Date"].max()

In [ ]:
df["Order Date"].dt.year.value_counts().sort_index()

In [ ]:
df[df["Order Date"].dt.year == 2024]

## Check 5 — Are missing values connected?
`crosstab` counts each combination of "title missing?" × "category missing?".

**Finding:** both missing together in 88,832 rows, alone only 626 / 908 →
one cause (most likely products no longer listed when the history was exported).

In [ ]:
missing = df[["Title", "Category", "Shipping Address State"]].isna()
pd.crosstab(missing["Title"], missing["Category"])

## Check 6 — Are there gaps over time?
**Findings:** no gaps; peaks every November–December; rise from 2020 (COVID),
maximum end of 2021, 2022 high and flat; the line drops in early 2023 because the data ends.
Late 2022 may be slightly undercounted: people who did the survey in Nov 2022
exported their history only up to that day.

In [ ]:
monthly = df.set_index("Order Date").resample("MS").size()
monthly.plot(figsize=(12, 4), title="Purchases per month")

## Check 7 — How do numbers relate to each other? (the matrix)
Correlation from −1 to +1 on a **monthly** table.

**Findings:** purchases ↔ spending 1.00, purchases ↔ customers 0.95, average price ↔ others ~0.1:
spending grew through **more items**, not higher prices.
**Caution:** correlation is not causation, and part of it reflects data coverage
(how many participants are in each month).

In [ ]:
m = df.set_index("Order Date").resample("MS").agg(
    purchases=("spend", "size"),
    spending=("spend", "sum"),
    avg_price=("Purchase Price Per Unit", "mean"),
    customers=("Survey ResponseID", "nunique"),
)
m.corr().round(2)

## Check 8 — Is the text clean?
To a computer, `"SHOES"`, `"shoes"` and `"SHOES "` are three different values.

**Findings:** no extra spaces; all states are 2-letter codes. Category uses **two naming
systems**: system codes (`SHOES`, `ABIS_BOOK`) and 680 human-readable names
(`Grocery`, `Paperback`, `Ecard Gift Certificate`). Keyword rules must compare in
upper case, and the gift card flag must catch both spellings.

In [ ]:
cat = df["Category"].dropna()
state = df["Shipping Address State"].dropna()

print("categories with extra spaces:", (cat != cat.str.strip()).sum())
print("categories not all uppercase:", (cat != cat.str.upper()).sum())
print("states with extra spaces:   ", (state != state.str.strip()).sum())
print("states not 2 letters:       ", (state.str.len() != 2).sum())

In [ ]:
odd = cat[cat != cat.str.upper()]
odd.value_counts().head(10)

## Summary
Almost nothing found was an error. Most findings are **real properties of how the data
was produced**: no order number, no payment method, export dates, two category systems,
participants joining at different times. The system that produced the data decides what
it can and cannot tell you.